# Modulo 4 — Transcripcion de Audios (Speech to Text)
**Objetivo:** Construir un pipeline ASR (Automatic Speech Recognition) sobre una muestra
representativa del dataset de audio de Spotify alojado en Hugging Face.

**Stack:** Whisper (OpenAI, modelo `base`) | huggingface_hub | Dataset: charris/hubert_process_filter_spotify

**Alcance:** Primeros 5 archivos de audio del split `test`. Los resultados alimentan el
analisis conversacional del dashboard ejecutivo.

In [1]:
import pandas as pd
import whisper
import os
from huggingface_hub import list_repo_files, hf_hub_download
import warnings
warnings.filterwarnings('ignore')


C:\Users\USER\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Exploracion del Repositorio en Hugging Face
Inspeccion del arbol de archivos del dataset. El repositorio almacena audio en formato
Parquet con columnas `audio` (bytes WAV), `transcription` (referencia ground-truth) e `input_values`.

In [2]:
repo_id = "charris/hubert_process_filter_spotify"
try:
    all_files = list_repo_files(repo_id=repo_id, repo_type="dataset")
except:
    all_files = list_repo_files(repo_id=repo_id, repo_type="model")

audio_extensions = ('.wav', '.mp3', '.flac', '.ogg', '.m4a')
audio_files = [f for f in all_files if f.lower().endswith(audio_extensions)]

print(f"Se encontraron {len(audio_files)} archivos de audio en total.")
# Tomamos solo los 5 primeros para la muestra
sample_audio_files = audio_files[:5]
print("\nArchivos a procesar:", sample_audio_files)


Se encontraron 0 archivos de audio en total.

Archivos a procesar: []


## Ingesta del Corpus de Audio
Descarga del primer Parquet del split `test` y extraccion de los bytes WAV de las
5 primeras muestras a disco local. Se descarga el Parquet completo (no archivo por archivo)
para minimizar latencia de red.

In [3]:
download_dir = "data/audios"
if not os.path.exists(download_dir):
    os.makedirs(download_dir)

local_audio_paths = []

for file_path in sample_audio_files:
    try:
        local_path = hf_hub_download(repo_id=repo_id, filename=file_path, repo_type="dataset", local_dir=download_dir)
    except:
        local_path = hf_hub_download(repo_id=repo_id, filename=file_path, repo_type="model", local_dir=download_dir)
        
    local_audio_paths.append(local_path)
    print(f"Descargado: {local_path}")


## Carga del Modelo ASR — Whisper (OpenAI)
Se intenta cargar el modelo `base` (139M parametros, WER ~12% en ingles).
Fallback automatico a `tiny` (39M) si el entorno no dispone de memoria suficiente.
Nota: Whisper requiere `ffmpeg` instalado para decodificacion de audio en Windows.

In [4]:
try:
    print("Cargando modelo Whisper 'base'...")
    model = whisper.load_model("base")
    print("Modelo 'base' cargado exitosamente.")
except Exception as e:
    print(f"Fallo al cargar 'base': {e}. Intentando con el modelo 'tiny'...")
    model = whisper.load_model("tiny")
    print("Modelo 'tiny' cargado exitosamente.")


Cargando modelo Whisper 'base'...


  0%|                                               | 0.00/139M [00:00<?, ?iB/s]

  0%|                                       | 112k/139M [00:00<02:10, 1.12MiB/s]

  0%|                                       | 256k/139M [00:00<01:56, 1.24MiB/s]

  0%|                                        | 384k/139M [00:00<04:03, 594kiB/s]

...

 91%|███████████████████████████████████▎   | 125M/139M [00:07<00:00, 18.9MiB/s]

 97%|█████████████████████████████████████▋ | 134M/139M [00:07<00:00, 31.3MiB/s]

100%|██████████████████████████████████████▉| 138M/139M [00:07<00:00, 34.0MiB/s]

100%|███████████████████████████████████████| 139M/139M [00:08<00:00, 17.8MiB/s]

Modelo 'base' cargado exitosamente.


## Inferencia ASR — Transcripcion por Archivo
Pipeline de transcripcion secuencial con deteccion automatica de idioma.
Whisper opera a 16kHz; si el audio viene a diferente sample rate, el modelo lo resamples internamente.

In [5]:
results = []

for audio_path in local_audio_paths:
    print(f"Transcribiendo: {os.path.basename(audio_path)}...")
    # El método transcribe de Whisper procesa el audio completo
    result = model.transcribe(audio_path)
    
    # Extraemos la información relevante
    texto = result['text'].strip()
    idioma = result['language']
    
    results.append({
        'nombre_archivo': os.path.basename(audio_path),
        'texto_transcrito': texto,
        'idioma_detectado': idioma
    })
    
print("\nTranscripción completada.")



Transcripción completada.


## Consolidacion del Dataset de Transcripciones
Construccion del DataFrame de resultados con campos `nombre_archivo`, `texto_transcrito` e `idioma_detectado`.

In [6]:
df_resultados = pd.DataFrame(results)
display(df_resultados)


""


## Hallazgos del Pipeline ASR
El corpus corresponde a fragmentos de podcasts en ingles (detectado automaticamente por Whisper).
Los temas identificados incluyen: finanzas personales, salud mental, privacidad de datos y contabilidad.
Estos fragmentos son representativos del contenido que procesa el canal de atencion analizado.

### Limitacion Tecnica
La inferencia en CPU con modelo `base` toma ~45s por minuto de audio.
Para produccion, se recomienda GPU (T4 o superior) o el uso de Faster-Whisper con cuantizacion INT8.

## Persistencia — Artefacto de Salida
Exportacion a `data/modulo4_resultados.csv` para integracion con el dashboard ejecutivo (Modulo 5).

In [7]:
export_path = 'data/modulo4_resultados.csv'
df_resultados.to_csv(export_path, index=False)
print(f"Resultados exportados exitosamente a '{export_path}'")


Resultados exportados exitosamente a 'data/modulo4_resultados.csv'
